# Mini-GPT: A Transformer Language Model From Scratch

*Source:* `~/Projects/school/tamu-grad/ecen740/ECEN740_project3.ipynb` (ECEN 740, Project 3).

This notebook builds a decoder-only transformer language model end to end on the
tiny-Shakespeare corpus: GPT-2-style byte-level **BPE** tokenization, **RMSNorm**,
**scaled dot-product** and **multi-head causal self-attention**, a **SwiGLU** feed-forward
block, a pre-norm transformer stack with tied embeddings, cross-entropy training, and
**temperature / top-p** sampling for generation. Each component ships with a small
self-grading test.


!!! note "Saved outputs are from the original GPU run"
    This notebook was **not** re-executed when it was added to the textbook. Training a
    transformer on CPU is impractical, so the cell outputs below are the ones saved from
    the original run on GPU (`DEVICE = 'cuda'`). The code is unchanged; to reproduce the
    numbers, run it on a CUDA machine.


# Build a Mini-GPT: Transformer Language Model from Scratch
### Machine Learning Course | Project 3

In this project you will implement the core components of a decoder-only Transformer language model and train it on real text to observe language generation.

Scaffolding, data loading, the training loop, and the full model assembly are provided. Your task is to fill in the `# YOUR CODE HERE` blocks.

## Grading

| Part | Component | Points |
|------|-----------|--------|
| 1A | BPE: pre-tokenisation | 10 |
| 1B | BPE: pair counting and merge loop | 10 |
| 1C | BPE: encoding with learned merges | 10 |
| 2A | RMSNorm | 10 |
| 2B | Scaled Dot-Product Attention | 15 |
| 2C | Multi-Head Self-Attention | 10 |
| 2D | SwiGLU Feed-Forward Network | 10 |
| 2E | Pre-Norm Transformer Block | 5 |
| 3A | Cross-Entropy Loss | 10 |
| 3B | Temperature and Top-p Decoding | 10 |
| **Total** | | **100** |
| Bonus | Short-answer reflection questions | 5 |

> Set Runtime > Change runtime type > T4 GPU before running. Use the sessions wisely.

In [1]:
!pip install regex -q

import math, re, time, json, os, random, collections, urllib.request
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict
import regex
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import types

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

_SCORES: Dict[str, Tuple[int, int]] = {}

def record_score(name: str, earned: int, max_pts: int):
    _SCORES[name] = (earned, max_pts)

def print_score_summary():
    print('\n' + '=' * 65)
    print(f'  {"Component":<44} {"Score":>10}')
    print('=' * 65)
    total_earned = total_max = 0
    for name, (earned, mx) in _SCORES.items():
        mark = 'PASS' if earned == mx else ('PARTIAL' if earned > 0 else 'FAIL')
        print(f'  {name:<44} {earned:>3}/{mx:<3}  [{mark}]')
        total_earned += earned; total_max += mx
    print('=' * 65)
    print(f'  {"TOTAL":<44} {total_earned:>3}/{total_max:<3}')
    print('=' * 65)
    print(f'  Bonus question worth 5 pts is graded separately by instructor.')
    print('=' * 65)


Device: cuda


---
## Dataset

We use **TinyShakespeare** (1 MB of Shakespeare plays). It is small enough to train in minutes on a free GPU, and its distinctive vocabulary makes it straightforward to judge whether the model has learned anything.

In [2]:
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_PATH = 'shakespeare.txt'
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
with open(DATA_PATH, 'r') as f:
    raw_text = f.read()
print(f'Dataset loaded: {len(raw_text):,} characters')
print(raw_text[:300])


Dataset loaded: 1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


---
## Part 1: Byte-Pair Encoding (BPE) Tokenizer

### Background

Neural networks require integer inputs. BPE is a subword tokenisation algorithm introduced by Sennrich et al. (2016) for machine translation and adopted by GPT-2, GPT-4, and LLaMA.

**The algorithm:**
1. Initialise the vocabulary with the 256 possible byte values.
2. Pre-tokenise the corpus into words using a regex.
3. Count adjacent token pairs across all words (weighted by frequency).
4. Merge the most frequent pair into a new token.
5. Repeat until the vocabulary reaches the desired size.

**Example:**
```
Corpus: low(5)  lower(2)  widest(3)  newest(6)
Round 1: most frequent pair (e, s) count 9  -> merged to 'es'
Round 2: most frequent pair (es, t)          -> merged to 'est'
```

**Recommended reading:**
- Sennrich et al. (2016): https://arxiv.org/abs/1508.07909
- Radford et al. (2019), GPT-2 Section 2.2: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
- OpenAI tiktoken source: https://github.com/openai/tiktoken

### Part 1A: Pre-tokenisation

The GPT-2 regex attaches leading whitespace to each word. In `'low low'`, matches are `'low'` and `' low'` (with leading space). These are **distinct** pre-tokens and must not be collapsed.

Implement `pretokenize(text)`:
1. Use `regex.finditer(GPT2_PAT, text)`.
2. Encode each match with UTF-8 and represent as a tuple of single-byte `bytes` objects.
3. Return a `Counter` of tuple frequencies.

In [5]:
GPT2_PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def pretokenize(text: str) -> collections.Counter:
    """
    Pre-tokenise with the GPT-2 regex.
    Returns Counter: tuple[bytes, ...] -> frequency.

    Note: 'low low' produces:
        (b'l',b'o',b'w')       : 1   (bare, no leading space)
        (b' ',b'l',b'o',b'w') : 1   (with leading space)
    """
    word_freq = collections.Counter()
    # YOUR CODE HERE
    for match in regex.finditer(GPT2_PAT, text):
        token = tuple(bytes([b]) for b in match.group().encode('utf-8'))
        word_freq[token] += 1
    return word_freq


In [6]:
# ── TEST 1A: Pre-tokenisation ─────────────────────────────────────────────────
def test_pretokenize():
    earned = 0; max_pts = 10

    try:
        wf = pretokenize('low low lower')
        key_bare  = tuple(bytes([b]) for b in 'low'.encode('utf-8'))
        key_space = tuple(bytes([b]) for b in ' low'.encode('utf-8'))
        key_lower = tuple(bytes([b]) for b in ' lower'.encode('utf-8'))
        assert wf[key_bare] + wf[key_space] == 2
        assert wf[key_lower] == 1
        earned += 2; print('[T1] Space-aware pre-tokenisation: PASS')
    except Exception as e: print(f'[T1] Space-aware: FAIL  {e}')

    try:
        wf = pretokenize('hello')
        key = tuple(bytes([b]) for b in 'hello'.encode('utf-8'))
        assert key in wf
        for b in key:
            assert isinstance(b, bytes) and len(b) == 1
        earned += 2; print('[T2] Byte representation: PASS')
    except Exception as e: print(f'[T2] Bytes: FAIL  {e}')

    try:
        wf = pretokenize('the apple')
        assert sum(wf.values()) == 2
        earned += 2; print('[T3] Word boundary separation: PASS')
    except Exception as e: print(f'[T3] Boundaries: FAIL  {e}')

    try:
        wf = pretokenize('caf\u00e9')
        assert all(isinstance(b, bytes) and len(b) == 1 for seq in wf for b in seq)
        earned += 2; print('[T4] Non-ASCII multi-byte characters: PASS')
    except Exception as e: print(f'[T4] Non-ASCII: FAIL  {e}')

    try:
        assert len(pretokenize('')) == 0
        earned += 2; print('[T5] Empty string: PASS')
    except Exception as e: print(f'[T5] Empty: FAIL  {e}')

    record_score('1A: Pre-tokenisation', earned, max_pts)
    print(f'\nScore 1A: {earned}/{max_pts}')

test_pretokenize()


[T1] Space-aware pre-tokenisation: PASS
[T2] Byte representation: PASS
[T3] Word boundary separation: PASS
[T4] Non-ASCII multi-byte characters: PASS
[T5] Empty string: PASS

Score 1A: 10/10


### Part 1B: BPE Training

Implement three functions:

**`get_pair_counts(corpus)`**: Count all adjacent pairs weighted by word frequency.

**`apply_merge(corpus, pair, new_token)`**: Replace every occurrence of `pair` with `new_token`, greedy left-to-right. In `(a, b, a, b)`, merging `(a, b)` gives `(ab, ab)`.

**`train_bpe(text, vocab_size)`**:
1. Call `pretokenize` to build the initial corpus.
2. Initialise vocabulary with all 256 byte values.
3. While `len(vocab) < vocab_size` **and** pairs remain:
   a. Get pair counts. Break if empty.
   b. Best pair: highest frequency; tie-break by lexicographically largest pair: `max(pair_counts, key=lambda p: (pair_counts[p], p))`
   c. Merge: `new_token = best_pair[0] + best_pair[1]`
   d. Record merge, update vocab, apply to corpus.
4. Return `(vocab, merges)`.

**Note on early stopping**: if the corpus runs out of pairs before reaching `vocab_size`, the loop exits. The invariant that always holds is `len(merges) == len(vocab) - 256`.

In [7]:
def get_pair_counts(corpus: dict) -> collections.Counter:
    """Return Counter of adjacent pairs weighted by word frequency."""
    counts = collections.Counter()
    # YOUR CODE HERE
    for word, freq in corpus.items():
      for i in range(len(word) - 1):
          counts[(word[i], word[i+1])] += freq
    return counts

def apply_merge(corpus: dict, pair: tuple, new_token: bytes) -> dict:
    """Replace every occurrence of pair with new_token (greedy left-to-right)."""
    new_corpus = {}
    for word, freq in corpus.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_corpus[tuple(new_word)] = freq
    return new_corpus

def train_bpe(text: str, vocab_size: int) -> Tuple[dict, list]:
    """
    Train byte-level BPE.
    Returns (vocab: dict[int, bytes], merges: list[tuple[bytes, bytes]]).
    Invariant: len(merges) == len(vocab) - 256 always holds.
    """
    assert vocab_size >= 256
    # YOUR CODE HERE
    corpus = dict(pretokenize(text))
    vocab = {i: bytes([i]) for i in range(256)}
    merges = []
    while len(vocab) < vocab_size:
        pair_counts = get_pair_counts(corpus)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p))
        new_token = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        vocab[len(vocab)] = new_token
        corpus = apply_merge(corpus, best_pair, new_token)
    return vocab, merges


In [8]:
# ── TEST 1B: BPE Training ─────────────────────────────────────────────────────
def test_bpe_training():
    earned = 0; max_pts = 10

    try:
        corpus = {(b'l',b'o',b'w'): 5, (b'l',b'o',b'w',b'e',b'r'): 2}
        pc = get_pair_counts(corpus)
        assert pc[(b'l',b'o')] == 7 and pc[(b'o',b'w')] == 7
        assert pc[(b'w',b'e')] == 2 and pc[(b'e',b'r')] == 2
        earned += 2; print('[T1] get_pair_counts weighted counts: PASS')
    except Exception as e: print(f'[T1] get_pair_counts: FAIL  {e}')

    try:
        corpus = {(b'a',): 10, (b'b',): 5}
        assert len(get_pair_counts(corpus)) == 0
        earned += 1; print('[T2] Single-token words produce no pairs: PASS')
    except Exception as e: print(f'[T2] Single-token: FAIL  {e}')

    try:
        corpus = {(b'l',b'o',b'w'): 5, (b'l',b'o',b'w',b'e',b'r'): 2}
        nc = apply_merge(corpus, (b'l',b'o'), b'lo')
        assert nc.get((b'lo',b'w')) == 5
        assert nc.get((b'lo',b'w',b'e',b'r')) == 2
        earned += 2; print('[T3] apply_merge correctness: PASS')
    except Exception as e: print(f'[T3] apply_merge: FAIL  {e}')

    try:
        corpus = {(b'a',b'b',b'a',b'b'): 3}
        nc = apply_merge(corpus, (b'a',b'b'), b'ab')
        assert nc.get((b'ab',b'ab')) == 3
        earned += 1; print('[T4] apply_merge overlapping pairs: PASS')
    except Exception as e: print(f'[T4] Overlapping: FAIL  {e}')

    try:
        v, m = train_bpe('low low low lower widest newest', 270)
        assert len(v) == 270
        earned += 2; print('[T5] train_bpe reaches target vocab size: PASS')
    except Exception as e: print(f'[T5] Vocab size: FAIL  {e}')

    try:
        # Invariant holds regardless of whether corpus is exhausted early.
        # Use a corpus rich enough to reach 270.
        v, m = train_bpe('low low low lower widest newest', 270)
        inv_ok = len(m) == len(v) - 256
        assert inv_ok, (
            f'Invariant len(merges)==len(vocab)-256 broken: '
            f'len(merges)={len(m)}, len(vocab)-256={len(v)-256}. '
            f'Every merge must add exactly one entry to vocab.')
        earned += 2; print(f'[T6] Invariant len(merges)==len(vocab)-256: PASS')
    except Exception as e: print(f'[T6] Invariant: FAIL  {e}')

    record_score('1B: BPE Training', earned, max_pts)
    print(f'\nScore 1B: {earned}/{max_pts}')

test_bpe_training()


[T1] get_pair_counts weighted counts: PASS
[T2] Single-token words produce no pairs: PASS
[T3] apply_merge correctness: PASS
[T4] apply_merge overlapping pairs: PASS
[T5] train_bpe reaches target vocab size: PASS
[T6] Invariant len(merges)==len(vocab)-256: PASS

Score 1B: 10/10


### Part 1C: BPE Encoding

Implement the `encode` method. For each pre-token:
1. Represent it as a list of single-byte `bytes` objects.
2. Repeatedly find the pair with the **lowest merge rank** (earliest learned merge) and apply it.
3. Continue until no applicable merge exists.
4. Look up each token in `self.token2id`.

**Why priority order matters**: merge rank encodes frequency. Merging the most frequent pair first is the correct decoding policy.

In [9]:
class BPETokenizer:
    def __init__(self, vocab: dict, merges: list):
        self.vocab      = vocab
        self.token2id   = {v: k for k, v in vocab.items()}
        self.merges     = merges
        self.merge_rank = {pair: i for i, pair in enumerate(merges)}

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    def encode(self, text: str) -> List[int]:
        """Encode text to a list of integer token IDs."""
        ids = []
        # YOUR CODE HERE
        for match in regex.finditer(GPT2_PAT, text):
            tokens = [bytes([b]) for b in match.group().encode('utf-8')]
            while len(tokens) >= 2:
                pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
                best = min(pairs, key=lambda p: self.merge_rank.get(p, float('inf')))
                if best not in self.merge_rank:
                    break
                new_tokens, i = [], 0
                while i < len(tokens):
                    if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                        new_tokens.append(tokens[i] + tokens[i+1])
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens
            ids.extend(self.token2id[t] for t in tokens)
        return ids

    def decode(self, ids: List[int]) -> str:
        raw = b''.join(self.vocab[i] for i in ids)
        return raw.decode('utf-8', errors='replace')


In [10]:
# ── TEST 1C: BPE Encoding ─────────────────────────────────────────────────────
def test_bpe_encoding():
    earned = 0; max_pts = 10
    vocab, merges = train_bpe('low low low lower widest newest newest newest newest', 270)
    tok = BPETokenizer(vocab, merges)

    try:
        text = 'low'
        assert tok.decode(tok.encode(text)) == text
        earned += 2; print('[T1] Roundtrip decode(encode(text))==text: PASS')
    except Exception as e: print(f'[T1] Roundtrip: FAIL  {e}')

    try:
        assert len(tok.encode('newest')) < len('newest'.encode('utf-8'))
        earned += 2; print('[T2] Compression of frequent words: PASS')
    except Exception as e: print(f'[T2] Compression: FAIL  {e}')

    try:
        ids = tok.encode('low lower widest')
        assert all(isinstance(i, int) and 0 <= i < tok.vocab_size for i in ids)
        earned += 2; print('[T3] IDs within valid range: PASS')
    except Exception as e: print(f'[T3] Valid range: FAIL  {e}')

    try:
        text = 'low lower widest newest'
        assert tok.encode(text) == tok.encode(text)
        earned += 2; print('[T4] Determinism: PASS')
    except Exception as e: print(f'[T4] Determinism: FAIL  {e}')

    try:
        v2, m2 = train_bpe('ab ab ab ab ab bc bc bc bc bc abc', 260)
        t2 = BPETokenizer(v2, m2)
        assert t2.decode(t2.encode('abc')) == 'abc'
        earned += 2; print('[T5] Merge priority order: PASS')
    except Exception as e: print(f'[T5] Merge priority: FAIL  {e}')

    record_score('1C: BPE Encoding', earned, max_pts)
    print(f'\nScore 1C: {earned}/{max_pts}')

test_bpe_encoding()


[T1] Roundtrip decode(encode(text))==text: PASS
[T2] Compression of frequent words: PASS
[T3] IDs within valid range: PASS
[T4] Determinism: PASS
[T5] Merge priority order: PASS

Score 1C: 10/10


---
## Tokenizer Training (Provided)

In [11]:
VOCAB_SIZE = 1000
print('Training BPE tokenizer...')
t0 = time.time()
vocab, merges = train_bpe(raw_text, VOCAB_SIZE)
tokenizer = BPETokenizer(vocab, merges)
print(f'Done in {time.time()-t0:.1f}s  vocab_size={tokenizer.vocab_size}')
sample = 'To be, or not to be, that is the question.'
enc = tokenizer.encode(sample)
print(f'Input:   "{sample}"')
print(f'Decoded: "{tokenizer.decode(enc)}"')
all_ids = tokenizer.encode(raw_text)
split = int(0.9 * len(all_ids))
train_ids = np.array(all_ids[:split], dtype=np.int32)
val_ids   = np.array(all_ids[split:],  dtype=np.int32)
print(f'Train: {len(train_ids):,}  Val: {len(val_ids):,}')


Training BPE tokenizer...
Done in 27.8s  vocab_size=1000
Input:   "To be, or not to be, that is the question."
Decoded: "To be, or not to be, that is the question."
Train: 416,483  Val: 46,276


---
## Data Loading (Provided)

In [12]:
class TokenDataset(Dataset):
    def __init__(self, ids, context_length):
        self.ids = torch.tensor(ids, dtype=torch.long); self.ctx = context_length
    def __len__(self):  return len(self.ids) - self.ctx
    def __getitem__(self, idx): return self.ids[idx:idx+self.ctx], self.ids[idx+1:idx+self.ctx+1]

CONTEXT_LEN = 128; BATCH_SIZE = 64
train_ds = TokenDataset(train_ids, CONTEXT_LEN)
val_ds   = TokenDataset(val_ids,   CONTEXT_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
x, y = next(iter(train_loader))
print(f'Input shape: {x.shape}  Target shape: {y.shape}')


Input shape: torch.Size([64, 128])  Target shape: torch.Size([64, 128])


---
## Part 2: Transformer Architecture

```
Token IDs -> Embedding -> [TransformerBlock x N] -> RMSNorm -> Linear -> Logits

TransformerBlock (pre-norm):
  x -> RMSNorm -> MultiHeadSelfAttention -> residual add
  x -> RMSNorm -> SwiGLU FFN             -> residual add
```

**Recommended reading:**
- Vaswani et al. (2017): https://arxiv.org/abs/1706.03762
- Touvron et al. (2023), LLaMA: https://arxiv.org/abs/2302.13971
- The Illustrated Transformer: https://jalammar.github.io/illustrated-transformer/

In [13]:
@dataclass
class ModelConfig:
    vocab_size:     int   = VOCAB_SIZE
    context_length: int   = CONTEXT_LEN
    d_model:        int   = 192
    n_heads:        int   = 6
    n_layers:       int   = 6
    d_ff:           int   = 512
    dropout:        float = 0.1

CFG = ModelConfig()
print(CFG)


ModelConfig(vocab_size=1000, context_length=128, d_model=192, n_heads=6, n_layers=6, d_ff=512, dropout=0.1)


### Part 2A: RMSNorm

```
RMSNorm(x) = ( x / sqrt( mean(x^2) + eps ) ) * g
```

`g` is a learnable per-dimension scale parameter initialised to 1. There is no bias and no mean subtraction.

**Recommended reading:**
- Zhang and Sennrich (2019): https://arxiv.org/abs/1910.07467

In [14]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # YOUR CODE HERE: define self.weight as a learnable parameter
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # rms = sqrt( x.pow(2).mean(-1, keepdim=True) + eps )
        # return (x / rms) * self.weight
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


In [15]:
# ── TEST 2A: RMSNorm ──────────────────────────────────────────────────────────
def test_rmsnorm():
    earned = 0; max_pts = 10

    try:
        norm = RMSNorm(16); x = torch.randn(3, 5, 16)
        assert norm(x).shape == x.shape
        earned += 2; print('[T1] Output shape: PASS')
    except Exception as e: print(f'[T1] Shape: FAIL  {e}')

    try:
        norm = RMSNorm(64); x = torch.randn(4, 10, 64) * 5.0
        rms = norm(x).pow(2).mean(dim=-1).sqrt()
        assert torch.allclose(rms, torch.ones_like(rms), atol=1e-3)
        earned += 3; print('[T2] RMS equals 1 when weight=1: PASS')
    except Exception as e: print(f'[T2] RMS=1: FAIL  {e}')

    try:
        norm = RMSNorm(8)
        with torch.no_grad(): norm.weight.fill_(2.0)
        rms = norm(torch.randn(2, 8)).pow(2).mean(dim=-1).sqrt()
        assert torch.allclose(rms, torch.full_like(rms, 2.0), atol=1e-3)
        earned += 3; print('[T3] Learnable weight scaling: PASS')
    except Exception as e: print(f'[T3] Weight scaling: FAIL  {e}')

    try:
        out = RMSNorm(32)(torch.full((2, 32), 1e4))
        assert not (torch.any(torch.isnan(out)) or torch.any(torch.isinf(out)))
        earned += 2; print('[T4] Numerical stability: PASS')
    except Exception as e: print(f'[T4] Stability: FAIL  {e}')

    record_score('2A: RMSNorm', earned, max_pts)
    print(f'\nScore 2A: {earned}/{max_pts}')

test_rmsnorm()


[T1] Output shape: PASS
[T2] RMS equals 1 when weight=1: PASS
[T3] Learnable weight scaling: PASS
[T4] Numerical stability: PASS

Score 2A: 10/10


### Part 2B: Scaled Dot-Product Attention

```
Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) + M ) V
```

The `1/sqrt(d_k)` scaling prevents variance growth. The causal mask M prevents position i from attending to positions > i.

```
Causal mask (length 4):
q0 [  0    -inf  -inf  -inf ]
q1 [  0     0    -inf  -inf ]
q2 [  0     0     0    -inf ]
q3 [  0     0     0     0   ]
```

**Recommended reading:**
- Vaswani et al. (2017), Section 3.2: https://arxiv.org/abs/1706.03762

In [16]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
    dropout_p: float = 0.0,
    training: bool = True,
) -> torch.Tensor:
    """
    Args:
        q, k: (..., seq, d_k)
        v:    (..., seq, d_v)
        mask: optional additive mask; use -inf to block positions
    Returns: (..., seq_q, d_v)
    """
    d_k = q.size(-1)
    # YOUR CODE HERE
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    attn = F.softmax(scores, dim=-1)
    attn = F.dropout(attn, p=dropout_p, training=training)
    return attn @ v


In [17]:
# ── TEST 2B: Scaled Dot-Product Attention ─────────────────────────────────────
def test_attention():
    earned = 0; max_pts = 15

    try:
        q=torch.randn(2,4,8,16); k=torch.randn(2,4,8,16); v=torch.randn(2,4,8,32)
        assert scaled_dot_product_attention(q,k,v).shape==(2,4,8,32)
        earned+=2; print('[T1] Output shape: PASS')
    except Exception as e: print(f'[T1] Shape: FAIL  {e}')

    try:
        T=6; q=torch.randn(1,1,T,8); k=torch.randn(1,1,T,8)
        v=torch.eye(T).unsqueeze(0).unsqueeze(0)
        rs=scaled_dot_product_attention(q,k,v,dropout_p=0.0).sum(dim=-1)
        assert torch.allclose(rs,torch.ones_like(rs),atol=1e-4)
        earned+=3; print('[T2] Attention weights sum to 1: PASS')
    except Exception as e: print(f'[T2] Weights sum: FAIL  {e}')

    try:
        T=5; q=torch.randn(1,1,T,8); k=torch.randn(1,1,T,8)
        v=torch.eye(T).unsqueeze(0).unsqueeze(0)
        causal=torch.tril(torch.ones(T,T)).log()
        out=scaled_dot_product_attention(q,k,v,mask=causal,dropout_p=0.0)
        for i in range(T):
            for j in range(i+1,T):
                assert abs(out[0,0,i,j].item())<1e-6
        earned+=5; print('[T3] Causal mask blocks future positions: PASS')
    except Exception as e: print(f'[T3] Causal mask: FAIL  {e}')

    try:
        torch.manual_seed(42); d_k=64; T=8
        q=torch.randn(1,1,T,d_k); k=torch.randn(1,1,T,d_k)
        v=torch.eye(T).unsqueeze(0).unsqueeze(0)
        avg_max=scaled_dot_product_attention(q,k,v,dropout_p=0.0)[0,0].max(dim=-1).values.mean().item()
        assert avg_max<0.95, f'avg_max={avg_max:.4f} >= 0.95, likely missing 1/sqrt(d_k)'
        earned+=3; print(f'[T4] sqrt(d_k) scaling present (avg_max={avg_max:.3f}): PASS')
    except Exception as e: print(f'[T4] Scaling: FAIL  {e}')

    try:
        torch.manual_seed(1)
        q=torch.randn(1,1,10,8); k=torch.randn(1,1,10,8); v=torch.randn(1,1,10,8)
        o1=scaled_dot_product_attention(q,k,v,dropout_p=0.5,training=True)
        o2=scaled_dot_product_attention(q,k,v,dropout_p=0.5,training=True)
        assert not torch.allclose(o1,o2)
        earned+=2; print('[T5] Dropout stochasticity: PASS')
    except Exception as e: print(f'[T5] Dropout: FAIL  {e}')

    record_score('2B: Scaled Dot-Product Attention', earned, max_pts)
    print(f'\nScore 2B: {earned}/{max_pts}')

test_attention()


[T1] Output shape: PASS
[T2] Attention weights sum to 1: PASS
[T3] Causal mask blocks future positions: PASS
[T4] sqrt(d_k) scaling present (avg_max=0.381): PASS
[T5] Dropout stochasticity: PASS

Score 2B: 15/15


### Part 2C: Multi-Head Self-Attention

```
MHA(X) = Concat(head_1, ..., head_h) W_O
head_j = Attention(X W_Qj, X W_Kj, X W_Vj)
```

Fuse Q, K, V into a single `qkv_proj` of output dimension `3 * d_model`. Reshape to (B, n_heads, T, d_head), apply attention, then reshape back.

In [18]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads  = cfg.n_heads
        self.d_head   = cfg.d_model // cfg.n_heads
        self.d_model  = cfg.d_model
        self.drop_p   = cfg.dropout
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out_proj  = nn.Linear(cfg.d_model, cfg.d_model,     bias=False)
        causal = torch.tril(torch.ones(cfg.context_length, cfg.context_length))
        causal = causal.masked_fill(causal==0, float('-inf')).masked_fill(causal==1, 0.0)
        self.register_buffer('causal_mask', causal)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        # YOUR CODE HERE
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.d_model, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        out = scaled_dot_product_attention(
            q, k, v,
            mask=self.causal_mask[:T, :T],
            dropout_p=self.drop_p,
            training=self.training,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


In [19]:
# ── TEST 2C: Multi-Head Self-Attention ────────────────────────────────────────
def test_mha():
    earned = 0; max_pts = 10

    try:
        mha=MultiHeadSelfAttention(CFG); x=torch.randn(2,CONTEXT_LEN,CFG.d_model)
        assert mha(x).shape==x.shape
        earned+=2; print('[T1] Output shape: PASS')
    except Exception as e: print(f'[T1] Shape: FAIL  {e}')

    try:
        mha=MultiHeadSelfAttention(CFG); x=torch.randn(2,10,CFG.d_model)
        assert not torch.allclose(mha(x),x)
        earned+=2; print('[T2] Non-trivial output: PASS')
    except Exception as e: print(f'[T2] Non-trivial: FAIL  {e}')

    try:
        mha=MultiHeadSelfAttention(CFG); mha.eval()
        x1=torch.randn(1,10,CFG.d_model); x2=x1.clone(); x2[0,7,:]+=100.0
        with torch.no_grad(): o1,o2=mha(x1),mha(x2)
        for pos in range(7):
            assert torch.allclose(o1[0,pos],o2[0,pos],atol=1e-4)
        earned+=4; print('[T3] Causal property: PASS')
    except Exception as e: print(f'[T3] Causal: FAIL  {e}')

    try:
        mha=MultiHeadSelfAttention(CFG)
        assert mha(torch.randn(2,5,CFG.d_model)).shape==(2,5,CFG.d_model)
        earned+=2; print('[T4] Short sequence: PASS')
    except Exception as e: print(f'[T4] Short seq: FAIL  {e}')

    record_score('2C: Multi-Head Self-Attention', earned, max_pts)
    print(f'\nScore 2C: {earned}/{max_pts}')

test_mha()


[T1] Output shape: PASS
[T2] Non-trivial output: PASS
[T3] Causal property: PASS
[T4] Short sequence: PASS

Score 2C: 10/10


### Part 2D: SwiGLU Feed-Forward Network

```
SwiGLU(x) = ( x W1  *  SiLU(x W_gate) ) W2
SiLU(t)   = t * sigmoid(t)
```

The W_gate path produces a learned soft gate. SwiGLU uses three weight matrices; to keep parameter count comparable to a standard two-matrix FFN, `d_ff` is set to approximately 8/3 * d_model.

**Recommended reading:**
- Shazeer (2020): https://arxiv.org/abs/2002.05202
- Geva et al. (2021): https://arxiv.org/abs/2012.14913

In [20]:
class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        # YOUR CODE HERE: define w1, w_gate, w2, dropout
        self.w1     = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w2     = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop   = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        return self.drop(self.w2(self.w1(x) * F.silu(self.w_gate(x))))


In [21]:
# ── TEST 2D: SwiGLU FFN ───────────────────────────────────────────────────────
def test_swiglu():
    earned = 0; max_pts = 10

    try:
        ffn=SwiGLUFFN(CFG); x=torch.randn(3,10,CFG.d_model)
        assert ffn(x).shape==x.shape
        earned+=2; print('[T1] Output shape: PASS')
    except Exception as e: print(f'[T1] Shape: FAIL  {e}')

    try:
        ffn=SwiGLUFFN(CFG)
        with torch.no_grad(): ffn.w_gate.weight.fill_(0.0)
        assert torch.allclose(ffn(torch.randn(1,1,CFG.d_model)),torch.zeros(1,1,CFG.d_model),atol=1e-5)
        earned+=4; print('[T2] Zero gate produces zero output: PASS')
    except Exception as e: print(f'[T2] Zero gate: FAIL  {e}')

    try:
        ffn=SwiGLUFFN(CFG)
        params=sum(p.numel() for p in ffn.parameters())
        expected=CFG.d_model*CFG.d_ff*2+CFG.d_ff*CFG.d_model
        assert params==expected
        earned+=2; print('[T3] Parameter count: PASS')
    except Exception as e: print(f'[T3] Param count: FAIL  {e}')

    try:
        ffn=SwiGLUFFN(CFG)
        with torch.no_grad(): ffn.w_gate.weight.data=-torch.abs(ffn.w_gate.weight.data)*1000
        assert not torch.any(torch.isnan(ffn(torch.randn(1,1,CFG.d_model))))
        earned+=2; print('[T4] SiLU stability with large negative gate: PASS')
    except Exception as e: print(f'[T4] Stability: FAIL  {e}')

    record_score('2D: SwiGLU FFN', earned, max_pts)
    print(f'\nScore 2D: {earned}/{max_pts}')

test_swiglu()


[T1] Output shape: PASS
[T2] Zero gate produces zero output: PASS
[T3] Parameter count: PASS
[T4] SiLU stability with large negative gate: PASS

Score 2D: 10/10


### Part 2E: Pre-Norm Transformer Block

```
Pre-norm:   x = x + Attention(Norm(x))  ;  x = x + FFN(Norm(x))
Post-norm:  x = Norm(x + Attention(x))  ;  x = Norm(x + FFN(x))
```

Residual connections provide a direct gradient path through all layers.

**Recommended reading:**
- Xiong et al. (2020): https://arxiv.org/abs/2002.04745

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLUFFN(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


In [23]:
# ── TEST 2E: TransformerBlock ─────────────────────────────────────────────────
def test_transformer_block():
    earned = 0; max_pts = 5

    try:
        blk=TransformerBlock(CFG)
        assert blk(torch.randn(2,CONTEXT_LEN,CFG.d_model)).shape==(2,CONTEXT_LEN,CFG.d_model)
        earned+=1; print('[T1] Shape preserved: PASS')
    except Exception as e: print(f'[T1] Shape: FAIL  {e}')

    try:
        blk=TransformerBlock(CFG); x=torch.randn(2,10,CFG.d_model)
        assert not torch.allclose(blk(x),x)
        earned+=1; print('[T2] Non-trivial transform: PASS')
    except Exception as e: print(f'[T2] Non-trivial: FAIL  {e}')

    try:
        blk=TransformerBlock(CFG); blk.eval()
        x1=torch.randn(1,15,CFG.d_model); x2=x1.clone(); x2[0,10,:]+=1000.0
        with torch.no_grad(): o1,o2=blk(x1),blk(x2)
        for p in range(10):
            assert torch.allclose(o1[0,p],o2[0,p],atol=1e-3)
        earned+=3; print('[T3] Causal property propagated through block: PASS')
    except Exception as e: print(f'[T3] Causal: FAIL  {e}')

    record_score('2E: TransformerBlock', earned, max_pts)
    print(f'\nScore 2E: {earned}/{max_pts}')

test_transformer_block()


[T1] Shape preserved: PASS
[T2] Non-trivial transform: PASS
[T3] Causal property propagated through block: PASS

Score 2E: 5/5


---
## Full Transformer Language Model (Provided)

Weight tying: the token embedding matrix and LM head share parameters, reducing parameter count and ensuring consistency between input embedding and output scoring (Press and Wolf, 2016).

In [24]:
class TransformerLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size,     cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x   = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks: x = block(x)
        return self.lm_head(self.norm(x))

model = TransformerLM(CFG).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Model parameters: 2,873,280


---
## Part 3: Training Objective and Decoding

### Part 3A: Cross-Entropy Loss

```
L = -(1/N) * sum_i [ logits[i, y_i] - log( sum_j exp(logits[i, j]) ) ]
```

Use `F.log_softmax` for numerical stability (subtracts max internally).

**Recommended reading:**
- Goodfellow et al., Deep Learning, Section 6.2.1: https://www.deeplearningbook.org

In [25]:
def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Args:
        logits:  (B, T, V) raw model scores
        targets: (B, T)    ground-truth token IDs
    Returns: scalar mean cross-entropy loss
    """
    B, T, V = logits.shape
    # YOUR CODE HERE
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs_flat = log_probs.view(B * T, V)
    targets_flat = targets.view(B * T)
    return -log_probs_flat[torch.arange(B * T), targets_flat].mean()


In [26]:
# ── TEST 3A: Cross-Entropy Loss ───────────────────────────────────────────────
def test_loss():
    earned = 0; max_pts = 10; V = 50

    try:
        logits=torch.zeros(2,4,V); targets=torch.randint(0,V,(2,4))
        assert abs(cross_entropy_loss(logits,targets).item()-math.log(V))<1e-4
        earned+=3; print('[T1] Uniform logits produce log(V) loss: PASS')
    except Exception as e: print(f'[T1] Uniform loss: FAIL  {e}')

    try:
        logits=torch.full((1,3,V),-1e9); targets=torch.tensor([[2,10,30]])
        for t_idx,tok in enumerate(targets[0]): logits[0,t_idx,tok]=0.0
        assert cross_entropy_loss(logits,targets).item()<0.01
        earned+=3; print('[T2] Perfect logits produce near-zero loss: PASS')
    except Exception as e: print(f'[T2] Perfect logits: FAIL  {e}')

    try:
        loss=cross_entropy_loss(torch.full((1,2,V),1e4),torch.zeros(1,2,dtype=torch.long))
        assert not torch.isnan(loss) and not torch.isinf(loss)
        earned+=2; print('[T3] Numerical stability: PASS')
    except Exception as e: print(f'[T3] Stability: FAIL  {e}')

    try:
        torch.manual_seed(7)
        logits=torch.randn(3,5,V); targets=torch.randint(0,V,(3,5))
        assert torch.allclose(F.cross_entropy(logits.view(-1,V),targets.view(-1)),
                              cross_entropy_loss(logits,targets),atol=1e-5)
        earned+=2; print('[T4] Matches F.cross_entropy reference: PASS')
    except Exception as e: print(f'[T4] Reference: FAIL  {e}')

    record_score('3A: Cross-Entropy Loss', earned, max_pts)
    print(f'\nScore 3A: {earned}/{max_pts}')

test_loss()


[T1] Uniform logits produce log(V) loss: PASS
[T2] Perfect logits produce near-zero loss: PASS
[T3] Numerical stability: PASS
[T4] Matches F.cross_entropy reference: PASS

Score 3A: 10/10


### Part 3B: Temperature and Top-p Decoding

Temperature T divides logits before softmax:
```
p_i = softmax( logits / T )_i
```

Top-p (nucleus) sampling (Holtzman et al., 2020): keep the smallest set of tokens whose cumulative probability exceeds p, then sample from that set.

Implement `generate(model, tokenizer, prompt, max_new_tokens, temperature, top_p)`.

**Recommended reading:**
- Holtzman et al. (2020): https://arxiv.org/abs/1904.09751

In [27]:
@torch.no_grad()
def generate(
    model: nn.Module,
    tokenizer: BPETokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float  = 1.0,
    top_p: float        = 0.9,
) -> str:
    model.eval()
    ids = tokenizer.encode(prompt)
    ctx = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    for _ in range(max_new_tokens):
        ctx_trimmed = ctx[:, -CFG.context_length:]
        logits = model(ctx_trimmed)[0, -1]
        # YOUR CODE HERE
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        remove = cumulative > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        sorted_probs[remove] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        choice = torch.multinomial(sorted_probs, num_samples=1)
        next_id = sorted_idx[choice]
        ctx = torch.cat([ctx, next_id.view(1, 1)], dim=1)

    return tokenizer.decode(ctx[0].tolist()[len(ids):])


In [28]:
# ── TEST 3B: Decoding ─────────────────────────────────────────────────────────
def test_generate():
    earned = 0; max_pts = 10

    try:
        out=generate(model,tokenizer,'ROMEO',max_new_tokens=20,temperature=1.0,top_p=1.0)
        assert isinstance(out,str) and len(out)>0
        earned+=2; print('[T1] Returns non-empty string: PASS')
    except Exception as e: print(f'[T1] Basic generation: FAIL  {e}')

    try:
        ids20 =tokenizer.encode(generate(model,tokenizer,'To be',max_new_tokens=20, temperature=1.0,top_p=1.0))
        ids100=tokenizer.encode(generate(model,tokenizer,'To be',max_new_tokens=100,temperature=1.0,top_p=1.0))
        assert len(ids100)>len(ids20)
        earned+=2; print('[T2] More tokens generated when requested: PASS')
    except Exception as e: print(f'[T2] Token count: FAIL  {e}')

    try:
        torch.manual_seed(0)
        o1=generate(model,tokenizer,'The king',max_new_tokens=30,temperature=0.0001,top_p=1.0)
        torch.manual_seed(0)
        o2=generate(model,tokenizer,'The king',max_new_tokens=30,temperature=0.0001,top_p=1.0)
        assert o1==o2
        earned+=3; print('[T3] Near-zero temperature is deterministic: PASS')
    except Exception as e: print(f'[T3] Determinism: FAIL  {e}')

    try:
        outputs=[generate(model,tokenizer,'What is',max_new_tokens=20,temperature=1.5,top_p=1.0) for _ in range(5)]
        assert len(set(outputs))>1
        earned+=3; print('[T4] High temperature produces varied outputs: PASS')
    except Exception as e: print(f'[T4] Diversity: FAIL  {e}')

    record_score('3B: Temperature and Top-p Decoding', earned, max_pts)
    print(f'\nScore 3B: {earned}/{max_pts}')

test_generate()


[T1] Returns non-empty string: PASS
[T2] More tokens generated when requested: PASS
[T3] Near-zero temperature is deterministic: PASS
[T4] High temperature produces varied outputs: PASS

Score 3B: 10/10


---
## Training Loop (Provided)

Trains for up to 4 epochs with linear warmup, cosine decay, and early stopping. Early stopping restores the best checkpoint if validation loss stops improving.

In [29]:
def evaluate(model, loader, max_batches=50):
    model.eval()
    losses=[]
    with torch.no_grad():
        for i,(x,y) in enumerate(loader):
            if i>=max_batches: break
            x,y=x.to(DEVICE),y.to(DEVICE)
            losses.append(cross_entropy_loss(model(x),y).item())
    model.train()
    return sum(losses)/len(losses)

LR=5e-4; N_EPOCHS=4; WARMUP_STEPS=400; EVAL_EVERY=250; PATIENCE=3
total_steps=N_EPOCHS*len(train_loader)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.1)

def lr_lambda(step):
    if step<WARMUP_STEPS: return step/max(1,WARMUP_STEPS)
    progress=(step-WARMUP_STEPS)/max(1,total_steps-WARMUP_STEPS)
    return max(0.1,0.5*(1.0+math.cos(math.pi*progress)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
best_val_loss=float('inf'); patience_count=0; best_state=None
model.train(); step=0; stop_training=False

for epoch in range(1,N_EPOCHS+1):
    if stop_training: break
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad()
        loss=cross_entropy_loss(model(x),y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); scheduler.step(); step+=1
        if step%EVAL_EVERY==0:
            vl=evaluate(model,val_loader)
            print(f'Epoch {epoch}  step {step:5d}  train {loss.item():.4f}  val {vl:.4f}  lr {scheduler.get_last_lr()[0]:.2e}')
            if vl<best_val_loss:
                best_val_loss=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; patience_count=0
            else:
                patience_count+=1
                if patience_count>=PATIENCE:
                    print(f'Early stopping at step {step}.'); stop_training=True; break

if best_state: model.load_state_dict(best_state); print(f'Best model restored (val={best_val_loss:.4f}).')
print('Training complete.')


Epoch 1  step   250  train 4.6405  val 4.5137  lr 3.13e-04
Epoch 1  step   500  train 3.7041  val 3.7013  lr 5.00e-04
Epoch 1  step   750  train 3.1208  val 3.3110  lr 5.00e-04
Epoch 1  step  1000  train 2.9463  val 3.2268  lr 4.99e-04
Epoch 1  step  1250  train 2.7784  val 3.1493  lr 4.99e-04
Epoch 1  step  1500  train 2.6101  val 3.1629  lr 4.98e-04
Epoch 1  step  1750  train 2.5418  val 3.2069  lr 4.97e-04
Epoch 1  step  2000  train 2.3676  val 3.2748  lr 4.95e-04
Early stopping at step 2000.
Best model restored (val=3.1493).
Training complete.


---
## Text Generation

In [30]:
prompts = ['ROMEO:', 'HAMLET:', 'To be, or not to be,', 'The king hath']
for p in prompts:
    print(f'Prompt: {p!r}')
    print(generate(model,tokenizer,p,max_new_tokens=120,temperature=0.8,top_p=0.9))
    print()


Prompt: 'ROMEO:'

Signior Gremio, what traitors are at evils?

MERCUTIO:
O, that he that I would think it,
Richard, I will follow you in this month.

MERCUTIO:
I am the language of your good master
I warrant him, for young Edward's land;
For now it is a while, I will not call it.

JULIET:

Prompt: 'HAMLET:'

All pale, come; come, go, go to to

JULIET:
I have made a physician.

ROMEO:
Good night, you will I think to speak.

JULIET:
Away, my lord!

ROMEO:
I know thee, get young Peter; you shall not
Than how shall not that I did with thee.

ROMEO:
Alack, fool, gentle church.

ROMEO:
And

Prompt: 'To be, or not to be,'
 put
Methought to surfeit; till, is not deliver'd
From the winds of our reasons.

PAULINA:
Go, master,
Since you can have married us.

LEONTES:
Peace, for our growing?

PAULINA:
I have injustice:
His borne me, good Lord Hortensio;
Nor changes. What's my

Prompt: 'The king hath'
 slain to the cause of my heart,
And came to fly to dy my hands
As doth some consequence of my pat

In [31]:
for temp in [0.3, 0.7, 1.0, 1.4]:
    print(f'Temperature: {temp}')
    print(generate(model,tokenizer,'OPHELIA:',max_new_tokens=80,temperature=temp,top_p=0.9))
    print()


Temperature: 0.3

O, no, my lord, I'll not be so much
To make a pair of my tongue.

POLIXENES:
I'll not believe you.

CAMILLO:
I know you have been borne
To see your daughter, and nothing but I
As you have been

Temperature: 0.7

I will not behold the duke
May say she is a foreign of a few of chamber,
And draw it like, that may be no villain,
And changed on my liberty.

POLIXENES:
What, should I know,
And I have

Temperature: 1.0

To speak, my loyal fortune,
The capuct young Duke of Hereford, to France
Do deceive this carool; it should be,
And as you love I shall guess him as my sword
Do to drop their harl dayship in my

Temperature: 1.4

I cannot lie to the vooways of one, look shall three.
Whumst what cookherish danother miment,
Pleale sir!

PAULINA:
Unpot damnter'd with daying stand: intend me. A better night;
Whilst your



---
## Exploration 1: Attention Pattern Visualisation

One of the most illuminating properties of the Transformer is that different attention heads learn to attend to different structural patterns. Some heads track syntactic dependencies, others track positional proximity, and others attend to specific token types such as punctuation, proper nouns, or repeated words.

The cell below extracts attention weights from each head in the first three layers and renders them as heatmaps. Examine:
- Which heads are diffuse (attend broadly) vs sharp (attend to one or two positions)
- Whether any head consistently attends to the immediately preceding token (a near-diagonal pattern)
- Whether any head attends to punctuation or the beginning of the sequence

No additional implementation is required. The weights flow through the `scaled_dot_product_attention` function you wrote in Part 2B.

In [32]:
_attn_weights = {}
sample_prompt = 'HAMLET: To be, or not to be, that is the question.'
sample_ids    = tokenizer.encode(sample_prompt)
sample_tensor = torch.tensor(sample_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
T_vis = len(sample_ids)
tokens_display = [tokenizer.decode([i]) for i in sample_ids]

model.eval()
with torch.no_grad():
    for layer_idx, block in enumerate(model.blocks):
        def patched_forward(self, x, li=layer_idx):
            B, T, D = x.shape
            qkv = self.qkv_proj(x)
            q, k, v = qkv.split(self.d_model, dim=-1)
            q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
            k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
            v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
            scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_head)
            scores = scores + self.causal_mask[:T,:T]
            attn   = F.softmax(scores, dim=-1)
            _attn_weights[li] = attn.detach().cpu()
            out = (attn @ v).transpose(1,2).contiguous().view(B,T,D)
            return self.out_proj(out)
        block.attn.forward = types.MethodType(patched_forward, block.attn)
    _ = model(sample_tensor)

for block in model.blocks:
    block.attn.forward = types.MethodType(MultiHeadSelfAttention.forward, block.attn)

n_layers_vis = min(3, CFG.n_layers)
fig, axes = plt.subplots(n_layers_vis, CFG.n_heads, figsize=(CFG.n_heads*2.5, n_layers_vis*2.5))
fig.suptitle('Attention patterns (first 3 layers)', fontsize=13, y=1.02)
tick_labels = [t[:5] if len(t)>5 else t.replace(' ','_') for t in tokens_display]

for li in range(n_layers_vis):
    w = _attn_weights[li][0]
    for hi in range(CFG.n_heads):
        ax = axes[li,hi] if n_layers_vis>1 else axes[hi]
        ax.imshow(w[hi].numpy(), cmap='Blues', vmin=0, aspect='auto')
        ax.set_title(f'L{li} H{hi}', fontsize=8)
        ax.set_xticks(range(T_vis)); ax.set_xticklabels(tick_labels, rotation=90, fontsize=5)
        ax.set_yticks(range(T_vis)); ax.set_yticklabels(tick_labels, fontsize=5)

plt.tight_layout()
plt.savefig('attention_patterns.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: attention_patterns.png')


Saved: attention_patterns.png


---
## Exploration 2: Ablation Study

An ablation study removes or replaces one design decision at a time and measures the effect on validation loss. This is the most direct way to understand why each architectural choice exists.

Three variants are trained for 300 steps each (approximately 2 minutes total on T4):

| Variant | What changes |
|---------|-------------|
| No causal mask | All positions attend to all positions |
| LayerNorm instead of RMSNorm | Adds mean subtraction and a bias term |
| ReLU instead of SwiGLU | Replaces the gating mechanism with a simple rectification |

Because each variant starts from random initialisation and trains for only 300 steps, absolute loss values will be higher than the baseline. The informative quantity is the **relative ordering**.

In [33]:
def train_ablation(model_factory, n_steps=300):
    m = model_factory().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=5e-4, weight_decay=0.1)
    m.train()
    for step_i,(x,y) in enumerate(train_loader):
        if step_i>=n_steps: break
        x,y=x.to(DEVICE),y.to(DEVICE)
        opt.zero_grad(); cross_entropy_loss(m(x),y).backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
    val=evaluate(m,val_loader,max_batches=30)
    del m; (torch.cuda.empty_cache() if DEVICE=='cuda' else None)
    return val

# No causal mask variant
class NoCausalMHA(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.n_heads=cfg.n_heads; self.d_head=cfg.d_model//cfg.n_heads
        self.d_model=cfg.d_model; self.drop_p=cfg.dropout
        self.qkv_proj=nn.Linear(cfg.d_model,3*cfg.d_model,bias=False)
        self.out_proj=nn.Linear(cfg.d_model,cfg.d_model,bias=False)
    def forward(self,x):
        B,T,D=x.shape
        qkv=self.qkv_proj(x); q,k,v=qkv.split(self.d_model,dim=-1)
        q=q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v=v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        out=scaled_dot_product_attention(q,k,v,mask=None,dropout_p=self.drop_p if self.training else 0.0,training=self.training)
        return self.out_proj(out.transpose(1,2).contiguous().view(B,T,D))

class NoCausalBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model); self.attn=NoCausalMHA(cfg)
        self.norm2=RMSNorm(cfg.d_model); self.ffn=SwiGLUFFN(cfg)
    def forward(self,x):
        x=x+self.attn(self.norm1(x)); x=x+self.ffn(self.norm2(x)); return x

class ReLUFFN(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.w1=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.w2=nn.Linear(cfg.d_ff,cfg.d_model,bias=False)
        self.drop=nn.Dropout(cfg.dropout)
    def forward(self,x): return self.w2(self.drop(F.relu(self.w1(x))))

class ReLUBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model); self.attn=MultiHeadSelfAttention(cfg)
        self.norm2=RMSNorm(cfg.d_model); self.ffn=ReLUFFN(cfg)
    def forward(self,x):
        x=x+self.attn(self.norm1(x)); x=x+self.ffn(self.norm2(x)); return x

def make_lm(block_cls):
    cfg=CFG
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
            self.pos_emb=nn.Embedding(cfg.context_length,cfg.d_model)
            self.drop=nn.Dropout(cfg.dropout)
            self.blocks=nn.ModuleList([block_cls(cfg) for _ in range(cfg.n_layers)])
            self.norm=RMSNorm(cfg.d_model)
            self.lm_head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
            self.lm_head.weight=self.tok_emb.weight
            for m in self.modules():
                if isinstance(m,(nn.Linear,nn.Embedding)): nn.init.normal_(m.weight,std=0.02)
        def forward(self,idx):
            B,T=idx.shape; x=self.drop(self.tok_emb(idx)+self.pos_emb(torch.arange(T,device=idx.device)))
            for blk in self.blocks: x=blk(x)
            return self.lm_head(self.norm(x))
    return M

def make_layernorm_lm():
    cfg=CFG
    class LNBlock(nn.Module):
        def __init__(self,cfg):
            super().__init__()
            self.norm1=nn.LayerNorm(cfg.d_model); self.attn=MultiHeadSelfAttention(cfg)
            self.norm2=nn.LayerNorm(cfg.d_model); self.ffn=SwiGLUFFN(cfg)
        def forward(self,x):
            x=x+self.attn(self.norm1(x)); x=x+self.ffn(self.norm2(x)); return x
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
            self.pos_emb=nn.Embedding(cfg.context_length,cfg.d_model)
            self.drop=nn.Dropout(cfg.dropout)
            self.blocks=nn.ModuleList([LNBlock(cfg) for _ in range(cfg.n_layers)])
            self.norm=nn.LayerNorm(cfg.d_model); self.lm_head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
            self.lm_head.weight=self.tok_emb.weight
            for m in self.modules():
                if isinstance(m,(nn.Linear,nn.Embedding)): nn.init.normal_(m.weight,std=0.02)
        def forward(self,idx):
            B,T=idx.shape; x=self.drop(self.tok_emb(idx)+self.pos_emb(torch.arange(T,device=idx.device)))
            for blk in self.blocks: x=blk(x)
            return self.lm_head(self.norm(x))
    return M

print('Running ablations (approx. 2 min on T4)...')
baseline_val  = evaluate(model, val_loader, max_batches=30)
val_no_causal = train_ablation(make_lm(NoCausalBlock))
val_layernorm = train_ablation(make_layernorm_lm())
val_relu      = train_ablation(make_lm(ReLUBlock))

print()
print('=' * 55)
print(f'  {"Variant":<32} {"Val Loss":>8}')
print('=' * 55)
print(f'  {"Baseline (trained model)":<32} {baseline_val:>8.4f}')
print(f'  {"No causal mask (300 steps)":<32} {val_no_causal:>8.4f}')
print(f'  {"LayerNorm instead of RMSNorm":<32} {val_layernorm:>8.4f}')
print(f'  {"ReLU instead of SwiGLU":<32} {val_relu:>8.4f}')
print('=' * 55)
print()
print('Ablation variants train from scratch for 300 steps.')
print('Focus on relative ordering, not absolute values.')


Running ablations (approx. 2 min on T4)...

  Variant                          Val Loss
  Baseline (trained model)           3.0825
  No causal mask (300 steps)         3.7303
  LayerNorm instead of RMSNorm       3.8857
  ReLU instead of SwiGLU             3.8574

Ablation variants train from scratch for 300 steps.
Focus on relative ordering, not absolute values.


---
## Bonus Questions (5 points total, graded by instructor)

**Q1.** The causal mask sets future positions to negative infinity before the softmax. What would happen if you instead set those positions to a large but finite negative number such as -1000? Is there any scenario where this approximation could fail numerically, and what would the symptom look like in generated text?

**Q2.** The `1/sqrt(d_k)` scaling in attention is fixed. Suppose you replaced it with a learnable scalar parameter initialised to `1/sqrt(d_k)`. What are the risks? Under what conditions might the model learn to collapse this parameter toward zero or infinity?

**Q3.** BPE tokenises "the" as a single token but a rare technical word as many tokens. What are the downstream implications for how the Transformer processes each word? Consider attention span, positional encoding coverage, and the number of residual updates the word receives.

**Q4.** SwiGLU uses three weight matrices but `d_ff` is set to approximately 8/3 * d_model rather than 4 * d_model. Why? What would happen if you kept `d_ff = 4 * d_model` with SwiGLU?

**Q5 (challenge).** Read the RoPE paper (Su et al., 2021: https://arxiv.org/abs/2104.09864) and explain in 3 to 5 sentences: (a) how RoPE encodes position geometrically, (b) why it generalises better to sequences longer than those seen during training, and (c) what the required mathematical prerequisite concept is.

In [ ]:
answer_q1 = """Your answer here."""
answer_q2 = """Your answer here."""
answer_q3 = """Your answer here."""
answer_q4 = """Your answer here."""
answer_q5 = """Your answer here."""


---
## Final Score

Run after completing all parts.

In [34]:
print_score_summary()


  Component                                         Score
  1A: Pre-tokenisation                          10/10   [PASS]
  1B: BPE Training                              10/10   [PASS]
  1C: BPE Encoding                              10/10   [PASS]
  2A: RMSNorm                                   10/10   [PASS]
  2B: Scaled Dot-Product Attention              15/15   [PASS]
  2C: Multi-Head Self-Attention                 10/10   [PASS]
  2D: SwiGLU FFN                                10/10   [PASS]
  2E: TransformerBlock                           5/5    [PASS]
  3A: Cross-Entropy Loss                        10/10   [PASS]
  3B: Temperature and Top-p Decoding            10/10   [PASS]
  TOTAL                                        100/100
  Bonus question worth 5 pts is graded separately by instructor.
